In [1]:
# Imports and Configurations
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from statsmodels.tsa.stattools import coint, adfuller
from statsmodels.regression.linear_model import OLS 
from statsmodels.tools import add_constant
from itertools import combinations
import warnings
warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

np.random.seed(42)

In [2]:
# Laading saved outputs from Notebook 2
sharpe_data = pd.read_csv('sharpe_data.csv', index_col=0, parse_dates=True)

cluster_k2 = pd.read_csv('cluster_assignments_sharpe_k2.csv')
cluster_k4 = pd.read_csv('cluster_assignments_sharpe_k4.csv')
cluster_k7 = pd.read_csv('cluster_assignments_sharpe_k7.csv')

print('Sharpe data shape:', sharpe_data.shape)
print('Date range:', sharpe_data.index[0].date(), 'to', sharpe_data.index[-1].date())
print('\nCluster assignment shapes:')
print(f' k=2: {cluster_k2.shape}, k=4:  {cluster_k4.shape}, k=7 = {cluster_k7.shape}')
print('\nSample cluster assignments (k=4, first 5 rows):')
print(cluster_k4.head())

Sharpe data shape: (470, 77)
Date range: 2014-12-31 to 2023-12-27

Cluster assignment shapes:
 k=2: (77, 3), k=4:  (77, 3), k=7 = (77, 3)

Sample cluster assignments (k=4, first 5 rows):
  Ticker  Cluster       Sector
0   AAPL        2   Technology
1   ADBE        1   Technology
2    ADI        1   Technology
3    ADP        4  Industrials
4   ALGN        3   Healthcare


In [3]:
# Redownloading price data
import yfinance as yf

tickers = sharpe_data.columns.tolist()

print(f'Re-downloading price data for {len(tickers)} stocks...')
print('Using same configuration as Notebook 2: weekly Wednesday, 2014-2023\n')

raw = yf.download(
    tickers,
    start='2014-01-01',
    end='2024-01-01',
    interval='1wk', 
    auto_adjust=True,
    progress=True
)

price_data = raw['Close']
price_data = price_data.resample('W-WED').last()
price_data = price_data.dropna(how='all')
price_data = price_data[tickers]
price_data = price_data.ffill().dropna(axis=1)

# Keeping only tickers present in both sharpe_data and price_data
common_tickers  = [t for t in tickers if t in price_data.columns]
price_data = price_data[common_tickers]

print(f'\nPrice data shape: {price_data.shape}')
print(f'Date range: {price_data.index[0].date()} to {price_data.index[-1].date()}')
print(f'Tickers retained: {len(common_tickers)}')

print(sharpe_data.columns.tolist() == common_tickers)

Re-downloading price data for 77 stocks...
Using same configuration as Notebook 2: weekly Wednesday, 2014-2023



[*********************100%***********************]  77 of 77 completed



Price data shape: (522, 77)
Date range: 2014-01-01 to 2023-12-27
Tickers retained: 77
True


In [4]:
# Building cluster lookup

cluster_map = {}

for k, df in [(2, cluster_k2), (4, cluster_k4), (7, cluster_k7)]:
    cluster_map[k] = {}
    for cluster_num in sorted(df['Cluster'].unique()):
        tickers_in_cluster = df[df['Cluster'] == cluster_num]['Ticker'].tolist()
        cluster_map[k][cluster_num] = tickers_in_cluster

for k in [2, 4, 7]:
    print(f'\nk={k}:')
    for cluster_num, tickers in cluster_map[k].items():
        print(f' Cluster {cluster_num} ({len(tickers)} stocks): {', '.join(tickers)}')


k=2:
 Cluster 1 (38 stocks): ADBE, ADI, AMAT, AMD, AMZN, ASML, BKNG, BKR, CCEP, CDNS, CPRT, FANG, FAST, GOOG, GOOGL, IDXX, INTU, KLAC, LIN, LRCX, MCHP, MELI, META, MRVL, MSFT, MU, NFLX, NVDA, ODFL, ON, PCAR, QCOM, SNPS, TSLA, VRSK, VRTX, WBD, WDAY
 Cluster 2 (39 stocks): AAPL, ADP, ALGN, AMGN, AVGO, AZN, BIIB, CDW, COST, CSCO, CSGP, CTAS, CTSH, DLTR, DXCM, EA, ENPH, EXC, FTNT, GILD, HON, ILMN, ISRG, KDP, MAR, MDLZ, MNST, MTCH, ORLY, PANW, PAYX, PEP, REGN, ROST, SBUX, SIRI, SMCI, TXN, XEL

k=4:
 Cluster 1 (38 stocks): ADBE, ADI, AMAT, AMD, AMZN, ASML, BKNG, BKR, CCEP, CDNS, CPRT, FANG, FAST, GOOG, GOOGL, IDXX, INTU, KLAC, LIN, LRCX, MCHP, MELI, META, MRVL, MSFT, MU, NFLX, NVDA, ODFL, ON, PCAR, QCOM, SNPS, TSLA, VRSK, VRTX, WBD, WDAY
 Cluster 2 (11 stocks): AAPL, AVGO, COST, CTSH, GILD, MAR, MNST, ORLY, PANW, REGN, SMCI
 Cluster 3 (11 stocks): ALGN, AZN, CSGP, DXCM, ENPH, HON, ISRG, MDLZ, MTCH, SBUX, SIRI
 Cluster 4 (17 stocks): ADP, AMGN, BIIB, CDW, CSCO, CTAS, DLTR, EA, EXC, FTNT, ILM

In [5]:
def run_adf(series, name=''):
    """
    Runs ADF test on a price series and returns a results dictionary. 
    H0: the series has a unit root (is non-stationary, I(1))
    We want to fail to reject H0 here - I(1) is the precondition for cointegration.
    """
    result = adfuller(series.dropna(), autolag='AIC')
    return  {
        'Ticker': name,
        'ADF Statistic': round(result[0], 4),
        'p-value': round(result[1], 4),
        'Lags Used': result[2],
        'I(1)': 'Yes' if result[1] > 0.05 else 'No'
    }

# Running ADF on every stock's price series
print('Running ADF unit root tests on all 77 price series...')
print('H0: series has a unit root (non-stationary)')
print('We want p > 0.05 to confirm I(1) - the cointegration precondition\n')

adf_results = []
for ticker in price_data.columns:
    adf_results.append(run_adf(price_data[ticker], name=ticker))

adf_df = pd.DataFrame(adf_results).set_index('Ticker')

# Summarizing ADF Results
i1_count = (adf_df['I(1)'] == 'Yes').sum()
i0_count = (adf_df['I(1)'] == 'No').sum()

print(f'Results: {i1_count} stocks confirmed I(1), {i0_count} stocks flagged as I(0)\n')

print('Stocks flagged as potential I(0) (Stationary - inspect carefully):')
print(adf_df[adf_df['I(1)'] == 'No'][['ADF Statistic', 'p-value']])

print("\nFull ADF results:")
print(adf_df.sort_values('p-value', ascending=False))

Running ADF unit root tests on all 77 price series...
H0: series has a unit root (non-stationary)
We want p > 0.05 to confirm I(1) - the cointegration precondition

Results: 76 stocks confirmed I(1), 1 stocks flagged as I(0)

Stocks flagged as potential I(0) (Stationary - inspect carefully):
        ADF Statistic  p-value
Ticker                        
BIIB          -3.7026   0.0041

Full ADF results:
        ADF Statistic  p-value  Lags Used I(1)
Ticker                                        
AVGO           2.9582   1.0000          4  Yes
SNPS           3.0976   1.0000         14  Yes
KLAC           2.9532   1.0000         19  Yes
PANW           2.0974   0.9988         16  Yes
PCAR           1.9953   0.9987          1  Yes
...               ...      ...        ...  ...
EA            -2.0993   0.2448          0  Yes
GILD          -2.4119   0.1384          0  Yes
CTSH          -2.4597   0.1256          0  Yes
WBD           -2.5763   0.0980          4  Yes
BIIB          -3.7026   0.0041 

In [7]:
# Running Engle-Granger Cointegration tests on all pairs within each cluster
def test_cointegration(s1, s2, ticker1, ticker2):
    """
    Runs Engle-Granger cointegration test on two price series.
    H0: no cointegration exists between the two series.
    We want to reject the H0 here - low p-value means cointegrated.
    """
    # Aligning series on shared dates
    combined = pd.concat([s1, s2], axis=1).dropna()
    s1_clean = combined.iloc[:,0]
    s2_clean = combined.iloc[:,1]

    score, pvalue,  _ = coint(s1_clean, s2_clean)
    pvalue = float(pvalue)
    score = float(score)
    assert isinstance(pvalue, float), f'pvalue is {type(pvalue)}'
    assert pvalue >= 0 and pvalue <= 1, f'pvalue out of range: {pvalue}'

    pvalue_rounded = round(float(pvalue), 4)
    score_rounded = round(float(score), 4)


    return {
        'Ticker 1': ticker1, 
        'Ticker 2': ticker2, 
        'Score': score_rounded,
        'p-value': pvalue_rounded,
        'Cointegrated': 'Yes' if pvalue < 0.05 else 'No'
    }

# Running Tests across all k values and clusters
all_coint_results = {}

for k in [2,4,7]:
    all_coint_results[k] = {}
    total_pairs = 0
    total_cointegrated = 0

    print(f'\n{'='*65}')
    print(f'k={k} INTRA-CLUSTER COINTEGRATION RESULTS:')
    print(f'\n{'='*65}')

    for cluster_num, tickers in cluster_map[k].items():
        
        # Filtering for tickers present in price data
        valid_tickers = [t for t in tickers if t in price_data.columns]

        # Generating all unique pairs
        pairs = list(combinations(valid_tickers, 2))
        results = []

        for t1, t2 in pairs:
            result = test_cointegration(
                price_data[t1], price_data[t2], t1, t2
            )
            results.append(result)

        # Converting to DataFrames and sorting by p-vaue
        results_df = pd.DataFrame(results).sort_values('p-value')
        all_coint_results[k][cluster_num] = results_df

        # Counting cointegrated pairs
        n_coint = (results_df['Cointegrated'] == 'Yes').sum()
        total_pairs += len(pairs)
        total_cointegrated += n_coint

        print(f"\nCluster {cluster_num} ({len(valid_tickers)} stocks,"
              f" {len(pairs)} pairs): {n_coint} cointegrated")

        # Printing top 5 pairs by p-value
        print(f" Top 5 pairs by p-value:")
        top5 = results_df.head(5).reset_index(drop=True)
        for i in range(len(top5)):
            row = top5.iloc[i]
            print(f"  {row['Ticker 1']}/{row['Ticker 2']}: "
                  f"p={row['p-value']:.4f} -> {row['Cointegrated']}")

    print(f"\nk={k} Summary: {total_cointegrated} cointegrated"
          f"  pairs out of {total_pairs} total")


k=2 INTRA-CLUSTER COINTEGRATION RESULTS:


Cluster 1 (38 stocks, 703 pairs): 76 cointegrated
  Raw Cointegrated column: ['Yes', 'Yes', 'Yes', 'Yes', 'Yes', 'Yes', 'Yes', 'Yes', 'Yes', 'Yes', 'Yes', 'Yes', 'Yes', 'Yes', 'Yes', 'Yes', 'Yes', 'Yes', 'Yes', 'Yes', 'Yes', 'Yes', 'Yes', 'Yes', 'Yes', 'Yes', 'Yes', 'Yes', 'Yes', 'Yes', 'Yes', 'Yes', 'Yes', 'Yes', 'Yes', 'Yes', 'Yes', 'Yes', 'Yes', 'Yes', 'Yes', 'Yes', 'Yes', 'Yes', 'Yes', 'Yes', 'Yes', 'Yes', 'Yes', 'Yes', 'Yes', 'Yes', 'Yes', 'Yes', 'Yes', 'Yes', 'Yes', 'Yes', 'Yes', 'Yes', 'Yes', 'Yes', 'Yes', 'Yes', 'Yes', 'Yes', 'Yes', 'Yes', 'Yes', 'Yes', 'Yes', 'Yes', 'Yes', 'Yes', 'Yes', 'No', 'Yes', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No', 'No'